## Past_Paper_ข้อ_1_to_4

In [1]:
import pandas as pd  
import pyproj
import pygeodesy
from pygeodesy.ellipsoidalKarney import LatLon
from pygeodesy import Ellipsoids
import math
from pyproj import Proj

#### ข้อ 1

In [2]:
# lat, lon, h = 13.73591343, 100.53392453, 74.245
# find UTM zone 45 N, ECEF, Height above MSL

# UTM zone 45 N
def lla2utm(lat,lon,ell_h,zone):
    lla = "epsg:4326"
    utm = "epsg:326" + str(zone)
    transproj = pyproj.Transformer.from_crs(lla, utm, always_xy=True)
    E, N, h = transproj.transform(lon, lat, ell_h, radians = False)
    return E, N, h

point_UTM = lla2utm(13.73591343, 100.53392453, 74.245, 45)
E = point_UTM[0]
N = point_UTM[1]
print(f"Easting: {E:.4f} m, Northing: {N:.4f} m")

Easting: 1975483.1158 m, Northing: 1560472.3626 m


In [3]:
# ECEF
def LLA2XYZ(lat, lon, ell_h):
    lla = "epsg:4326"
    ecef = '+proj=geocent +ellps=WGS84'
    transproj = pyproj.Transformer.from_crs(lla, ecef, always_xy=True)
    x,y,z = transproj.transform(lon, lat, ell_h, radians = False)
    return x, y, z

point_ECEF = LLA2XYZ(13.73591343, 100.53392453, 74.245)
X = point_ECEF[0]
Y = point_ECEF[1]
Z = point_ECEF[2]
print(f"X: {X:.4f} m, Y: {Y:.4f} m, Z: {Z:.4f} m")

X: -1132915.0947 m, Y: 6092528.5142 m, Z: 1504633.1345 m


In [4]:
# find Height above MSL
def Geoid_seperation(lat, lon):
    geoid_model = r"C:\Lab_Geodetic_Survey\Midterm_Prepare\tgm2017-1_2.pgm"
    geoid_interpolator = pygeodesy.GeoidKarney(geoid_model)
    pos = LatLon(lat, lon)
    N = geoid_interpolator(pos)
    return N

N = Geoid_seperation(13.73591343, 100.53392453)
ellipsoidal_height = 74.245
orthometric_height = ellipsoidal_height - N
print(f"Height above MSL: {orthometric_height:.4f} m")

Height above MSL: 104.6873 m


#### ข้อ 2

In [5]:
def LLC2UTM(e, n, h):
    llc_proj = '+proj=lcc +lon_0=99 +lat_0=0 +lat_1=35 +lat_2=85 +x_0=500000 +y_0=0 +ellps=WGS84'
    utm_proj = '+proj=utm +zone=46 +south +ellps=WGS84'

    transproj = pyproj.Transformer.from_crs(llc_proj, utm_proj, always_xy=True)
    E, N, H = transproj.transform(e, n, h)

    return E, N, H

LLC_point = LLC2UTM(544528.4768, 1521137.0493, 15.236)
print(f"UTM Easting: {LLC_point[0]:.4f} m, UTM Northing: {LLC_point[1]:.4f} m, Height: {LLC_point[2]:.4f} m")

UTM Easting: 1193654.4264 m, UTM Northing: 11115699.4714 m, Height: 15.2360 m


#### ข้อ 3

In [6]:
# lat, lon, h = 13.73591343, 100.53392453, 500.434
# ground distace = 150.345 m
# find ellipsoidal distance

ground_distance = 150.345

def get_Rm_for_LDP(lat):
    #Mean Radius of Curvature
    e = Ellipsoids.WGS84
    return e.rocGauss(lat)

R = get_Rm_for_LDP(13.73591343)
ESF = (R) / ((R) + (500.434)) # อย่าลืม convert m to km

ellipsoidal_distance = ground_distance * ESF
print(f"Ellipsoidal distance: {ellipsoidal_distance:.4f} m")

Ellipsoidal distance: 150.3332 m


In [7]:
# find grid distance (Lambert Conformal Conic)
def LLC_scale_factor(lat,lon):
    llc_proj = '+proj=lcc +lon_0=99 +lat_0=0 +lat_1=35 +lat_2=85 +x_0=500000 +y_0=0 +ellps=WGS84'
    prj = pyproj.Proj(llc_proj)
    fc = prj.get_factors(lon,lat,radians=False)
    scaleX = fc.parallel_scale
    scaleY = fc.meridional_scale
    scaleArea = fc.areal_scale 
    mer_conv = fc.meridian_convergence
    return scaleX, scaleY, scaleArea, mer_conv

scaleX, scaleY, scaleArea, mer_conv = LLC_scale_factor(13.73591343, 100.53392453)
print(f"Scale factor X: {scaleX:.6f}, Scale factor Y: {scaleY:.6f}, Scale factor Area: {scaleArea:.6f}, Meridian convergence: {mer_conv:.6f} degrees")

Scale factor X: 1.221166, Scale factor Y: 1.221166, Scale factor Area: 1.491247, Meridian convergence: 1.386903 degrees


In [8]:
grid_distance = ground_distance * ESF * scaleX
print(f"Grid distance: {grid_distance:.4f} m")

Grid distance: 183.5818 m


#### ข้อ 4

In [9]:
# อย่าลืม k = 1 / ESF
def get_Rm_for_LDP(lat):
    #Mean Radius of Curvature
    e = Ellipsoids.WGS84
    return e.rocGauss(lat)

R = get_Rm_for_LDP(16.8)
R

6360309.281284429

In [10]:
# คิดไม่ออก เอา max,min มาเฉลี่ยเลย
h = (645.01 + 151.99) / 2
h

398.5

In [11]:
ESF = (R) / ((R) + (h)) # อย่าลืม convert m to km
ESF

0.9999373497394154

In [12]:
print(1/ESF)

1.0000626541858857


##### 4.1

In [13]:
tm = 'proj=tmerc +lat_0=0 +lon_0=102.200 +k = 1.0000626541858857 +x_0=500000 +y_0=0 +ellps=WGS84 +units=m +no_defs'
# proj=tmerc -> กำหมดใช้ projection แบบ Transverse Mercator
# +lat_0=0 กำหนดละติจูดเริ่มต้นที่ Equator
# +lon_0=102.200 กำหนด central meridian ของ projection ซึ่งอยู่จุดกึ่งกลางของพื้นที่ที่ต้องการทำแผนที่
# +k = 1.0000626541858857 -> ค่า scale factor ที่ central meridian คำนวนจาก k0 = (R+h)/R ให้ลอยสูงขึ้นมาเกินทรงรีเท่ากับระดับความ h ที่กำหนดไว้
# +x_0=500000 -> กำหนดค่า false easting ของ projection ป้องกันไม่ให้ค่าพิกัดเป็นลบ
# +y_0=0 -> กำหนดค่า false northing ของ projection เป็น 0 เนื่องจากพื้นที่อยู่ซีกโลกเหนือ
# +ellps=WGS84 -> ใช้ทรงรี WGS84 เป็น reference ellipsoid
# +units=m -> กำหนดหน่วยเป็นเมตร
# +no_defs -> ไม่ใช้ค่า default ที่ติดมากับ pyproj

##### 4.2

##### เพราะแก้ปัญหาความคาดเคลื่อนของระยะทางในงานวิศวกรรมสำรวจ เนื่องจาก UTM เป็นการฉายแผนที่ไม่ได้อยู่ในระดับความสูงเดียวกับพื้นที่จริง ทำให้ grid distance ไม่เท่ากับ ground distance ซึ่งเป็นความคาดเคลื่อนที่เกิดจาก GSF และ ESF ดังนั้นการออกแบบ LDP เป็นการสร้างระบบพิกัดเฉพาะสำหรับพื้นที่นั้นๆ โดยปรับยก projection surface ให้สอดคล้องกับความสูงเฉลี่ยของผิว Terrain ของหน้างาน เพื่อบังคับให้ CSF มีค่าน้อยที่สุด เข้าใกล้ 1.000000 ให้มากที่สุด


##### 4.3

In [14]:
tm = 'proj=tmerc +lat_0=0 +lon_0=102.200 +k = 1.0000626541858857 +x_0=500000 +y_0=0 +ellps=WGS84 +units=m +no_defs'

lat = 16.3785234
lon = 101.8304747
h = 100.434

ldp_proj = Proj(tm)
E,N = ldp_proj(lon, lat)
h_tm = h

print(f"Easting: {E:.4f} m, Northing: {N:.4f} m, Height: {h_tm:.4f} m")

Easting: 460520.7026 m, Northing: 1811680.2775 m, Height: 100.4340 m
